# 📈 02. Phân Loại Danh Mục Hàng Hóa Theo Ma Trận ABC/XYZ & Hoạch Định Tồn Kho
**Dự án**: Streaming MLOps E-Commerce — Dự Báo Nhu Cầu Hàng Hóa
**Giai đoạn**: Tuần 4 — Phân loại hàng hóa & Chiến lược tồn kho

---
### Cơ sở lý thuyết quản trị chuỗi cung ứng:
1. **Phân tích ABC (Nguyên lý Pareto 80/15/5)**: Đánh giá tầm quan trọng của sản phẩm dựa trên **Doanh thu đóng góp tích lũy**.
   - **Nhóm A (High Value)**: 20% SKU mang lại 80% doanh thu.
   - **Nhóm B (Moderate Value)**: 30% SKU mang lại 15% doanh thu tiếp theo.
   - **Nhóm C (Low Value / Long Tail)**: 50% SKU còn lại chỉ mang lại 5% doanh thu.
2. **Phân tích XYZ (Hệ số biến thiên nhu cầu - Coefficient of Variation CV)**:
   $$CV = \frac{\sigma}{\mu} = \frac{\text{Độ lệch chuẩn nhu cầu}}{\text{Nhu cầu trung bình}}$$
   - **Nhóm X ($CV \le 0.5$)**: Nhu cầu cực kỳ ổn định, dự báo rất chính xác.
   - **Nhóm Y ($0.5 < CV \le 1.0$)**: Nhu cầu biến động vừa phải (theo mùa vụ/khuyến mãi).
   - **Nhóm Z ($CV > 1.0$)**: Nhu cầu biến động thất thường, gián đoạn (lumpy demand), khó dự báo.
3. **Ma trận 9 ô ABC/XYZ**: Kết hợp 2 chiều để tối ưu hóa chính sách tồn kho (Safety Stock, Reorder Point) và chỉ định mô hình Machine Learning phù hợp.


In [ ]:
# 1. Thiết lập môi trường và cấu hình hiển thị
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

sys.path.insert(0, os.path.abspath('..'))
from ml.features.abc_xyz import ABCXYZClassifier
from warehouse.etl.transform import unify_schema, clean_data


### 2. Tải và chuẩn bị dữ liệu bán hàng


In [ ]:
data_path = os.path.join('..', 'data', 'historical_orders_clean.parquet')
raw_path = os.path.join('..', 'data', 'historical_orders_raw.parquet')

if os.path.exists(data_path):
    clean_df = pd.read_parquet(data_path)
    print(f'✓ Đã tải {len(clean_df):,} dòng đơn hàng sạch.')
elif os.path.exists(raw_path):
    raw_df = pd.read_parquet(raw_path)
    clean_df = clean_data(unify_schema(raw_df))
    print(f'✓ Đã làm sạch {len(clean_df):,} dòng đơn hàng từ file raw.')
else:
    print('! Đang sinh dữ liệu lịch sử mẫu 60 ngày...')
    from scripts.simulate_historical_data import generate_historical_orders
    raw_df = generate_historical_orders(days=60, base_orders_per_day=200)
    clean_df = clean_data(unify_schema(raw_df))

print(f'Tổng số SKU trong danh mục: {clean_df["sku"].nunique()}')


### 3. Thực hiện Phân loại ABC/XYZ & Tính toán Chỉ số Tồn kho
Sử dụng module `ABCXYZClassifier` để tính toán doanh thu tích lũy, hệ số biến thiên $CV$, Safety Stock ($SS$) và Reorder Point ($ROP$).


In [ ]:
# Khởi tạo bộ phân loại với Lead time = 3 ngày và Service Level = 95% (Z = 1.65)
classifier = ABCXYZClassifier(
    pareto_a=0.80,
    pareto_b=0.95,
    cv_x=0.50,
    cv_y=1.00,
    default_lead_time_days=3,
    default_service_level=0.95,
)

matrix_df = classifier.fit_transform(clean_df)
summary = classifier.get_summary_report(matrix_df)

print('Bảng mẫu kết quả phân loại (Top 10 SKU theo doanh thu):')
display(matrix_df[['sku', 'product_name', 'total_revenue', 'rev_share', 'cum_rev_share', 'abc_class', 'cv', 'xyz_class', 'matrix_class', 'safety_stock', 'reorder_point']].head(10))


### 4. Trực quan hóa Đường cong Pareto (Phân loại ABC)
Đường cong Lorenz thể hiện mức độ tập trung doanh thu theo nguyên lý 80/20.


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

sorted_skus = matrix_df.sort_values(by='total_revenue', ascending=False).reset_index(drop=True)
x = np.arange(len(sorted_skus))

# Cột doanh thu từng SKU
ax1.bar(x, sorted_skus['total_revenue'] / 1e6, color='#4A90E2', alpha=0.7, label='Doanh thu từng SKU (Triệu VND)')
ax1.set_xlabel('Thứ hạng SKU')
ax1.set_ylabel('Doanh thu (Triệu VND)', color='#4A90E2')
ax1.tick_params(axis='y', labelcolor='#4A90E2')

# Đường tích lũy doanh thu
ax2 = ax1.twinx()
ax2.plot(x, sorted_skus['cum_rev_share'] * 100, color='#D0021B', linewidth=2.5, label='Doanh thu tích lũy (%)')
ax2.axhline(80, color='orange', linestyle='--', label='Ngưỡng Nhóm A (80%)')
ax2.axhline(95, color='green', linestyle='--', label='Ngưỡng Nhóm B (95%)')
ax2.set_ylabel('Tỷ lệ tích lũy (%)', color='#D0021B')
ax2.set_ylim(0, 105)
ax2.tick_params(axis='y', labelcolor='#D0021B')

plt.title('Đường cong Pareto: Phân bố Doanh thu tích lũy theo SKU (Phân loại ABC)')
fig.tight_layout()
plt.show()

for k, v in summary['abc_breakdown'].items():
    print(f'Nhóm {k}: {v["sku_count"]} SKUs ({v["sku_pct"]}%) đóng góp {v["revenue_pct"]}% tổng doanh thu.')


### 5. Trực quan hóa Ma trận 9 ô ABC/XYZ (9-Box Matrix Heatmap)
Ma trận kết hợp 2 chiều: Giá trị kinh tế (ABC) và Mức độ biến động nhu cầu (XYZ).


In [ ]:
# Lập bảng pivot 3x3 đếm số lượng SKU trong từng ô
pivot_counts = matrix_df.pivot_table(index='abc_class', columns='xyz_class', values='sku', aggfunc='count', fill_value=0)
pivot_counts = pivot_counts.reindex(index=['A', 'B', 'C'], columns=['X', 'Y', 'Z'], fill_value=0)

# Lập bảng pivot 3x3 tính tổng doanh thu trong từng ô
pivot_rev = matrix_df.pivot_table(index='abc_class', columns='xyz_class', values='total_revenue', aggfunc='sum', fill_value=0)
pivot_rev_pct = (pivot_rev / matrix_df['total_revenue'].sum() * 100).reindex(index=['A', 'B', 'C'], columns=['X', 'Y', 'Z'], fill_value=0).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pivot_counts, annot=True, fmt='d', cmap='YlGnBu', cbar=False, ax=axes[0], annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('Số lượng SKU theo ma trận 9 ô ABC/XYZ')
axes[0].set_xlabel('Mức độ biến động nhu cầu (XYZ)')
axes[0].set_ylabel('Đóng góp doanh thu (ABC)')

sns.heatmap(pivot_rev_pct, annot=True, fmt='.1f', cmap='OrRd', cbar=False, ax=axes[1], annot_kws={'size': 14, 'weight': 'bold'})
axes[1].set_title('Tỷ trọng Doanh thu (%) theo ma trận 9 ô ABC/XYZ')
axes[1].set_xlabel('Mức độ biến động nhu cầu (XYZ)')
axes[1].set_ylabel('Đóng góp doanh thu (ABC)')

plt.tight_layout()
plt.show()


### 6. Chiến lược Quản trị Tồn kho & Lựa chọn Mô hình Dự báo
Dựa trên ma trận 9 ô, thiết lập ma trận chiến lược chuỗi cung ứng và kiến trúc MLOps:

| Phân khúc | Chiến lược Quản trị Tồn kho | Chính sách Safety Stock | Mô hình ML khuyến nghị |
|---|---|---|---|
| **AX** | Just-In-Time (JIT), kiểm soát hàng ngày | Thấp (Nhu cầu ổn định) | **LightGBM / Ridge Regression** (độ chính xác cao) |
| **AY** | Dự báo theo sự kiện, chuẩn bị đệm trước Flash Sale | Trung bình (Đệm theo tuần) | **LightGBM / XGBoost** có lag & cờ khuyến mãi |
| **AZ** | Sản phẩm giá trị cao rủi ro lớn; đệm Safety Stock cao | Cao (Phòng ngừa đứt hàng) | **Deep Learning (LSTM/GRU)** hoặc Two-Stage |
| **BX** | Điểm đặt hàng lại tự động (ROP) | Trung bình | **LightGBM Baseline** |
| **BY** | Đặt hàng theo lô kinh tế (EOQ) kết hợp mùa vụ | Trung bình - Cao | **LightGBM / Random Forest** |
| **BZ** | Kiểm soát chặt chẽ quy mô đặt hàng | Thấp - Trung bình | **XGBoost** kết hợp phân loại xác suất mua |
| **CX** | Đặt hàng lô lớn định kỳ để giảm chi phí giao dịch | Thấp | **Moving Average / Exponential Smoothing** |
| **CY** | Tồn kho tối thiểu, đặt hàng khi có nhu cầu | Tối thiểu | **Mô hình Heuristic giản đơn** |
| **CZ** | Hàng đuôi dài; cân nhắc Make-to-Order hoặc Dropship | Rất thấp (Tránh ứ đọng vốn) | **Croston's Method** (Intermittent Demand) |


In [ ]:
# Trực quan hóa mối quan hệ giữa Doanh thu và Hệ số biến thiên CV kèm quy mô Safety Stock
plt.figure(figsize=(12, 7))
scatter = sns.scatterplot(
    data=matrix_df,
    x='cv',
    y=matrix_df['total_revenue'] / 1e6,
    hue='matrix_class',
    size='safety_stock',
    sizes=(50, 400),
    palette='tab10',
    alpha=0.85,
)
plt.axvline(0.50, color='gray', linestyle='--', alpha=0.7, label='Ngưỡng X/Y (CV=0.5)')
plt.axvline(1.00, color='red', linestyle='--', alpha=0.7, label='Ngưỡng Y/Z (CV=1.0)')
plt.xlabel('Hệ số biến thiên nhu cầu (CV = std / mean)')
plt.ylabel('Tổng doanh thu (Triệu VND)')
plt.title('Bản đồ phân tán SKU: CV vs Doanh thu (Kích thước bóng = Mức Safety Stock)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 7. Xuất kết quả phân loại & Hoàn thiện Tuần 4
Lưu kết quả phân loại ma trận ABC/XYZ và tham số tồn kho để các module phục vụ (Model Serving / FastAPI) ở Tuần 7 truy xuất trực tiếp.


In [ ]:
out_path = os.path.join('..', 'data', 'abc_xyz_inventory_policy.csv')
matrix_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'✓ Đã lưu bảng chính sách tồn kho và phân loại ABC/XYZ ra: {out_path}')

# Thống kê 5 sản phẩm nhóm A cần ưu tiên dự báo cao nhất:
top_a = matrix_df[matrix_df['abc_class'] == 'A'][['sku', 'product_name', 'matrix_class', 'avg_daily_demand', 'safety_stock', 'reorder_point', 'recommended_model']]
print('\nDanh sách các sản phẩm Nhóm A ưu tiên hàng đầu:')
display(top_a)


### 8. Kết luận & Chuyển giao sang Tuần 5
- **Phát hiện chính**: Ma trận ABC/XYZ phân hóa rõ nét danh mục sản phẩm. Nhóm sản phẩm **AX và AY** là trụ cột doanh thu cần tập trung tối ưu hóa độ chính xác mô hình ML, trong khi nhóm **Z** cần chính sách đệm tồn kho (Safety Stock) để chống đứt gãy nguồn cung.
- **Kết nối Tuần 5**: Bảng phân loại này sẽ được dùng để gán nhãn trọng số mẫu (sample weights) và lựa chọn thuật toán huấn luyện thích hợp khi thiết lập MLflow Tracking Server và Feature Store.
